In [ ]:
# Cell 1: Imports

import mailbox
import email
import re
import json 
import pandas as pd
import random
import time
from pathlib import Path
from bs4 import BeautifulSoup 
from tqdm.notebook import tqdm
from mlx_lm import load, generate
from collections import Counter


PROJECT = Path.home() / "llm-mail-trainer"
MBOX_FILE = PROJECT / "data/raw/All mail Including Spam and Trash-002.mbox"

print(f"MBOX file exists: {MBOX_FILE.exists()}")
print(f"File size: {MBOX_FILE.stat().st_size / 1e9:.2f} GB")

# Cell 2: Open MBOX and count emails

mbox = mailbox.mbox(str(MBOX_FILE))
total_emails = len(mbox)
print(f"Total emails in mailbox: {total_emails:,}")

# Cell 3: Look at one raw email
sample = mbox[0]

print("===EMAIL KEYS===")
print(list(sample.keys()))

print("\n=== SUBJECT ===")
print(sample['subject'])

print("\n=== FROM ===")
print(sample['from'])

print("\n=== DATE ===")
print(sample['date'])

print("\n=== CONTENT TYPE ===")
print(sample.get_content_type())

# Cell 4: Extract body from email

def get_body(message):
    """Extract text from email body."""

    if message.is_multipart():
        # Email has multiple parts (text + html + attachments)
        for part in message.walk():
            ctype = part.get_content_type()
            if ctype == 'text/plain':
                payload = part.get_payload(decode=True)
                if payload:
                    return payload.decode('utf-8', errors='ignore')
            elif ctype == 'text/html':
                payload = part.get_payload(decode=True)
                if payload:
                    soup = BeautifulSoup(payload.decode('utf-8', errors='ignore'),'lxml')
                    return soup.get_text(separator=' ',strip= True)
    else:
        # Single part email
        payload = message.get_payload(decode=True)
        if payload:
            text = payload.decode('utf-8', errors='ignore')
            if message.get_content_type() == 'text/html':
                soup = BeautifulSoup(text, 'lxml')
                return soup.get_text(separator=' ',strip= True)
            return text

    return ''

# Test on sample email
body = get_body(sample)
print(f"Body length: {len(body)} characters")
print(f"\n=== FIRST 500 CHARS ===\n{body[:500]}")

# Cell 5: Clean text function

def clean_text(text):
    """Remove noise from email text."""

    # Remove URLs
    text = re.sub(r'http[s]?://\S+', '', text)

    # Remove email addresses
    text = re.sub(r'\S+@\S+\.\S+', '', text)

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text)
    
    # Remove very long strings (encoded data)
    text = re.sub(r'\S{80,}', '', text)
    
    return text.strip()

# Test
cleaned = clean_text(body)
print(f"Before cleaning: {len(body)} chars")
print(f"After cleaning: {len(cleaned)} chars")
print(f"\n=== CLEANED TEXT ===\n{cleaned}")

# Cell 6: Decode encoded headers (like Subject, From)
def decode_header(header):
    """Decode email header that may be encoded."""
    if header is None:
        return ''
    
    try:
        decoded_parts = email.header.decode_header(header)
        result = []
        for content, charset in decoded_parts:
            if isinstance(content, bytes):
                content = content.decode(charset or 'utf-8', errors='ignore')
            result.append(str(content))
        return ' '.join(result)
    except Exception:
        return str(header)

# Test on the HDFC email
hdfc_email = mbox[27]

print("=== DECODED SUBJECT ===")
print(decode_header(hdfc_email['subject']))

print("\n=== DECODED FROM ===")
print(decode_header(hdfc_email['from']))

# Cell 7: Complete single email parser
def parse_email(message):
    """Parse a single email into clean structured data."""

    body = get_body(message)
    body = clean_text(body)

     # Skip if body too short
    if len(body) < 50:
        return None

    return {
        'subject' : clean_text(decode_header(message['subject'])),
        'sender' : clean_text(decode_header(message['from'])),
        'date': message['date'] or '',
        'body' : body[:5000]
        
    }

# Test on HDFC email
result = parse_email(hdfc_email)

print("=== PARSED EMAIL ===")
for key, value in result.items():
    if key == 'body':
        print(f"{key}: {value[:200]}...")
    else:
        print(f"{key}: {value}")

# Cell 8: Parse all emails
parsed_emails = []
failed = 0

print("Parsing all the mails ...")
for i in tqdm(range(total_emails), desc="Processing"):
    try:
        msg = mbox[i]
        result = parse_email(msg)
        if result:
            result['id'] = len(parsed_emails)
            parsed_emails.append(result)
    except Exception as e:
        failed += 1
        continue

print(f"\n✅ Successfully parsed: {len(parsed_emails):,}")
print(f"❌ Failed/skipped: {failed + (total_emails - len(parsed_emails) - failed):,}")
print(f"📊 Success rate: {len(parsed_emails)/total_emails*100:.1f}%")

# Cell 9: Save parsed emails to JSON
output_path = PROJECT / "data/parsed/emails.json"
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(parsed_emails, f, ensure_ascii=False)

# Verify
file_size = output_path.stat().st_size / 1e6
print(f"✅ Saved to: {output_path}")
print(f"📁 File size: {file_size:.1f} MB")
print(f"📧 Total emails: {len(parsed_emails):,}")

# Cell 10: Data summary
import pandas as pd

df = pd.DataFrame(parsed_emails)

print("=== DATA SUMMARY ===")
print(f"Total emails: {len(df):,}")
print(f"\nColumns: {list(df.columns)}")

print(f"\n=== BODY LENGTH STATS ===")
df['body_length'] = df['body'].str.len()
print(df['body_length'].describe())

print(f"\n=== TOP 10 SENDERS ===")
print(df['sender'].value_counts().head(10))

# Cell 11: Load parsed emails from cache

cache_path = PROJECT / "data/parsed/emails.json"

with open(cache_path, 'r', encoding='utf-8') as f:
    parsed_emails = json.load(f)

print(f"✅ Loaded {len(parsed_emails):,} emails from cache")

# Cell 12: Random sampling
random.seed(42)

# Pick 500 random emails
sample_size = 500
sample_emails = random.sample(parsed_emails,sample_size)

print(f"Total emails: {len(parsed_emails):,}")
print(f"Sample size: {len(sample_emails)}")

# Preview one sample
print(f"\n=== SAMPLE EMAIL #1 ===")
print(f"Subject: {sample_emails[0]['subject']}")
print(f"Sender: {sample_emails[0]['sender']}")
print(f"Body: {sample_emails[0]['body'][:300]}...")

# Cell 13: Classification prompt template

CLASSIFICATION_PROMPT = """You are an email classifier. Analyze this email and categorize it.

EMAIL:
Subject: {subject}
From: {sender}
Body: {body}

TASK:
Classify this email into exactly ONE category.

CATEGORIES:
- finance: Banks, payments, transactions, investments, credit cards, loans, UPI, wallets
- shopping: Orders, deliveries, purchases, e-commerce
- social: Social networks, personal messages, invitations
- work: Job-related, recruitment, office, meetings, projects
- newsletter: Digests, subscriptions, blogs, articles
- promotional: Marketing, offers, discounts, advertisements
- other: Anything that doesn't fit above

OUTPUT FORMAT (JSON only, no other text):
{{"category": "<category>", "confidence": "<high/medium/low>", "reason": "<brief 5-10 word reason>"}}
"""

def build_prompt(email_data):
    """Build classification prompt for one email."""
    return CLASSIFICATION_PROMPT.format(
        subject=email_data['subject'][:200],
        sender=email_data['sender'][:100],
        body=email_data['body'][:2000]
    )

# Test: See what prompt looks like
test_prompt = build_prompt(sample_emails[0])
print(f"Prompt length: {len(test_prompt)} characters")
print(f"\n=== PROMPT PREVIEW ===\n{test_prompt[:1000]}...")

# Cell 14: Load Phi-3 model
model_path = str(PROJECT / "models/base/phi3-mini")

print("Loading Phi-3 model...")
model, tokenizer = load(model_path)
print("✅ Model loaded")

# Cell 15: Test classification on one email
test_email = sample_emails[0]

# Build prompt
prompt = build_prompt(test_email)

# Send to Phi-3
print("Classifying email...")
print(f"Subject: {test_email['subject'][:80]}...")
print("-" * 50)

response = generate(
    model, 
    tokenizer, 
    prompt=prompt,
    max_tokens=100,
    verbose=False
)

print(f"\n=== PHI-3 RESPONSE ===\n{response}")

# Cell 16: JSON extraction helper

def extract_json(response):
    """Extract JSON object from LLM response."""

    # Find JSON pattern in response
    match = re.search(r'\{[^{}]*\}', response)

    if(match):
          try:
              return json.loads(match.group())
          except json.JSONDecodeError:
              return None
    return None

# Test on previous response
parsed = extract_json(response)

print("=== EXTRACTED JSON ===")
print(parsed)
print(f"\nCategory: {parsed['category']}")
print(f"Confidence: {parsed['confidence']}")
print(f"Reason: {parsed['reason']}")

# Cell 17: Classify all sample emails
results = []
failed = 0

print(f"Classifying {len(sample_emails)} emails...")
print("Estimated time: ~5 minutes\n")

start_time = time.time()

for i, email_data in enumerate(tqdm(sample_emails, desc="Classifying")):
    try:
        # Build prompt
        prompt = build_prompt(email_data)
        
        # Get classification
        response = generate(
            model, 
            tokenizer, 
            prompt=prompt,
            max_tokens=100,
            verbose=False
        )
        
        # Extract JSON
        parsed = extract_json(response)
        
        if parsed:
            results.append({
                'id': email_data.get('id', i),
                'subject': email_data['subject'],
                'sender': email_data['sender'],
                'category': parsed.get('category', 'other'),
                'confidence': parsed.get('confidence', 'low'),
                'reason': parsed.get('reason', '')
            })
        else:
            failed += 1
            
    except Exception as e:
        failed += 1
        continue

elapsed = time.time() - start_time

print(f"\n✅ Classified: {len(results)}")
print(f"❌ Failed: {failed}")
print(f"⏱️ Time: {elapsed/60:.1f} minutes")
print(f"⚡ Speed: {len(results)/elapsed:.1f} emails/sec")

# Cell 18: Category distribution

categories = Counter([r['category'] for r in results])

print("=== CATEGORY DISTRIBUTION ===\n")
for category, count in categories.most_common():
    pct = count / len(results) * 100
    bar = "█" * int(pct / 2)
    print(f"{category:12} {count:4} ({pct:5.1f}%) {bar}")

print(f"\n📊 Total classified: {len(results)}")

# Cell 19: Save classification results
results_path = PROJECT / "data/parsed/classification_results.json"

with open(results_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"✅ Saved {len(results)} results to {results_path}")

# Cell 20: Extract finance emails

finance_results = [r for r in results if r['category'] == 'finance']

print(f"=== FINANCE EMAILS: {len(finance_results)} ===\n")

# Show first 10

for i, email in enumerate(finance_results[:10]):

    print(f"{i+1}. {email['subject'][:70]}")

    print(f"   Sender: {email['sender'][:50]}")

    print(f"   Reason: {email['reason']}")

    print()

# Cell 21: Get full details of finance emails
finance_results = [r for r in results if r['category'] == 'finance']

# Get full email data for finance emails
finance_ids = [r['id'] for r in finance_results]
finance_emails_full = [e for e in parsed_emails if e['id'] in finance_ids]

print(f"Finance emails with full body: {len(finance_emails_full)}")

# Show senders
print("\n=== FINANCE SENDERS ===")
senders = [e['sender'] for e in finance_emails_full]
for sender, count in Counter(senders).most_common(15):
    print(f"  {sender[:50]:50} : {count}")

    



            
            




/Users/ranjit/llm-mail-trainer/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


MBOX file exists: True
File size: 2.94 GB
Total emails in mailbox: 41,948
===EMAIL KEYS===
['X-GM-THRID', 'X-Gmail-Labels', 'Delivered-To', 'Received', 'X-Google-Smtp-Source', 'X-Received', 'ARC-Seal', 'ARC-Message-Signature', 'ARC-Authentication-Results', 'Return-Path', 'Received', 'Received-SPF', 'Authentication-Results', 'DKIM-Signature', 'DKIM-Signature', 'Received', 'Received', 'Content-Transfer-Encoding', 'Content-Type', 'Date', 'From', 'Mime-Version', 'Message-ID', 'Subject', 'Reply-To', 'Feedback-ID', 'List-Unsubscribe', 'List-Unsubscribe-Post', 'x-campaignid', 'X-SG-EID', 'X-SG-ID', 'To', 'X-Entity-ID']

=== SUBJECT ===
Update: Your secret santa is here

=== FROM ===
Internshala Trainings <trainings@mail.internshala.com>

=== DATE ===
Thu, 25 Dec 2025 12:02:27 +0000 (UTC)

=== CONTENT TYPE ===
text/html
Body length: 273 characters

=== FIRST 500 CHARS ===
Internshala Trainings Internshala (Scholiverse Educare Pvt. Ltd.) 901A and 901B, Iris Tech Park, Sector - 48, Sohna Road, G

Processing:   0%|          | 0/41948 [00:00<?, ?it/s]


✅ Successfully parsed: 40,820
❌ Failed/skipped: 1,128
📊 Success rate: 97.3%
✅ Saved to: /Users/ranjit/llm-mail-trainer/data/parsed/emails.json
📁 File size: 76.1 MB
📧 Total emails: 40,820
=== DATA SUMMARY ===
Total emails: 40,820

Columns: ['subject', 'sender', 'date', 'body', 'id']

=== BODY LENGTH STATS ===
count    40820.000000
mean      1654.802670
std       1373.202087
min         50.000000
25%        588.000000
50%       1225.000000
75%       2465.000000
max       5000.000000
Name: body_length, dtype: float64

=== TOP 10 SENDERS ===
sender
                       2235
Flipkart               1155
LinkedIn               1008
Medium Daily Digest     948
Quora Digest            914
Classroom | Scaler      831
Upwork Notification     700
Groww Digest            698
"Guruji"                678
"ICICI Bank"            629
Name: count, dtype: int64
✅ Loaded 40,820 emails from cache
Total emails: 40,820
Sample size: 500

=== SAMPLE EMAIL #1 ===
Subject: Capgemini Group - You have been subm

Classifying:   0%|          | 0/500 [00:00<?, ?it/s]